# MNIST Knowledge Distillation Walkthrough

This notebook is a step-by-step learning walkthrough. The repository's `.py` scripts, experiment log, result tables, and summaries are the authoritative sources for the controlled experiments.

Learning goals:

- load one MNIST batch
- load a fixed teacher checkpoint
- compare hard labels with teacher soft targets
- see what temperature changes
- calculate hard loss, soft loss, and combined KD loss
- update only the student model


## 1. Imports and Project Setup

This setup is portable. It works whether the notebook is opened from the repository root or from inside the `notebooks` folder.


In [1]:
from pathlib import Path
import sys

import torch
import torch.nn.functional as F
from torch.optim import Adam

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from kd_research.data import create_mnist_loaders
from kd_research.models import SimpleCNNStudentModel, SimpleCNNTeacherModel

print("project root detected:", PROJECT_ROOT.name)


project root detected: knowledge-distillation-research


## 2. Walkthrough Seed

This seed makes the small notebook demonstration easier to follow. It is not the full formal reproducibility protocol used for controlled experiments.


In [2]:
torch.manual_seed(0)
device = torch.device("cpu")
print("walkthrough seed set to 0")
print("device:", device)


walkthrough seed set to 0
device: cpu


## 3. Load One MNIST Batch

The KD idea still starts with normal training data: images and true labels.


In [3]:
train_loader, _ = create_mnist_loaders(
    data_dir=PROJECT_ROOT / "data",
    batch_size=4,
    download=False,
    num_workers=0,
)

images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)

print("image batch shape:", tuple(images.shape))
print("label batch shape:", tuple(labels.shape))
print("labels:", labels.tolist())


image batch shape: (4, 1, 28, 28)
label batch shape: (4,)
labels: [6, 8, 8, 7]


## 4. Create the Student Model

The student is the model we will actually update. In KD, the teacher provides extra learning signal, but the student is the model being trained.


In [4]:
student = SimpleCNNStudentModel().to(device)
print(student)


SimpleCNNStudentModel(
  (network): Sequential(
    (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Flatten(start_dim=1, end_dim=-1)
    (3): Linear(in_features=6272, out_features=10, bias=True)
  )
)


## 5. Load the Fixed Teacher Checkpoint

The teacher was trained earlier by the project scripts. Here we load it only so it can produce soft targets.


In [5]:
checkpoint_path = PROJECT_ROOT / "checkpoints" / "mnist_teacher_baseline_001.pt"
print("teacher checkpoint:", checkpoint_path.relative_to(PROJECT_ROOT))
print("checkpoint exists:", checkpoint_path.exists())

assert checkpoint_path.exists(), "Expected MNIST teacher checkpoint is missing."

teacher = SimpleCNNTeacherModel().to(device)
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False,
)
teacher.load_state_dict(checkpoint["model_state_dict"])
print("teacher checkpoint loaded")


teacher checkpoint: checkpoints\mnist_teacher_baseline_001.pt
checkpoint exists: True
teacher checkpoint loaded


## 6. Freeze the Teacher

The teacher should not learn during distillation. It acts like a fixed source of soft targets.


In [6]:
teacher.eval()

for parameter in teacher.parameters():
    parameter.requires_grad = False

trainable_teacher_parameters = sum(
    parameter.numel()
    for parameter in teacher.parameters()
    if parameter.requires_grad
)

print("trainable teacher parameters:", trainable_teacher_parameters)


trainable teacher parameters: 0


## 7. Teacher and Student Forward Passes

The teacher forward pass uses `torch.inference_mode()` because we do not need teacher gradients. The student forward pass keeps gradients available because the student will be updated.


In [7]:
with torch.inference_mode():
    teacher_logits = teacher(images)

student_logits = student(images)

print("teacher logits shape:", tuple(teacher_logits.shape))
print("student logits shape:", tuple(student_logits.shape))


teacher logits shape: (4, 10)
student logits shape: (4, 10)


## 8. Hard Labels and Soft Targets

Hard labels give one correct class. Teacher soft targets give a probability distribution over all classes. The non-target probabilities can carry extra information.


In [8]:
teacher_probs = F.softmax(teacher_logits, dim=1)

print("true labels:", labels.tolist())
print("teacher top prediction:", teacher_probs.argmax(dim=1).tolist())
print("teacher probabilities for first example:")
print(torch.round(teacher_probs[0], decimals=4))


true labels: [6, 8, 8, 7]
teacher top prediction: [6, 8, 8, 7]
teacher probabilities for first example:
tensor([0., 0., 0., 0., 0., 0., 1., 0., 0., 0.])


## 9. Temperature Demo

Higher temperature softens the teacher distribution. That means the largest class probability becomes less dominant, and the smaller non-target probabilities become easier to see.


In [9]:
teacher_probs_t1 = F.softmax(teacher_logits / 1.0, dim=1)
teacher_probs_t2 = F.softmax(teacher_logits / 2.0, dim=1)

print("first example, T=1:")
print(torch.round(teacher_probs_t1[0], decimals=4))
print("first example, T=2:")
print(torch.round(teacher_probs_t2[0], decimals=4))
print("max probability at T=1:", round(teacher_probs_t1[0].max().item(), 4))
print("max probability at T=2:", round(teacher_probs_t2[0].max().item(), 4))


first example, T=1:
tensor([0., 0., 0., 0., 0., 0., 1., 0., 0., 0.])
first example, T=2:
tensor([8.0000e-04, 0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e-04, 1.0000e-04,
        9.9870e-01, 0.0000e+00, 0.0000e+00, 1.0000e-04])
max probability at T=1: 1.0
max probability at T=2: 0.9987


## 10. Complete KD Loss

The hard loss uses the true labels. The soft loss compares the student distribution with the teacher distribution. The `temperature**2` factor keeps the soft-loss scale more comparable when temperature changes.

`alpha` controls the balance:

- higher `alpha`: more hard-label learning
- lower `alpha`: more teacher soft-target learning


In [10]:
def knowledge_distillation_loss(
    student_logits,
    teacher_logits,
    true_labels,
    temperature,
    alpha,
):
    student_log_probs = F.log_softmax(
        student_logits / temperature,
        dim=1,
    )
    teacher_probs = F.softmax(
        teacher_logits / temperature,
        dim=1,
    )

    soft_loss = (
        F.kl_div(
            student_log_probs,
            teacher_probs,
            reduction="batchmean",
        )
        * temperature**2
    )

    hard_loss = F.cross_entropy(
        student_logits,
        true_labels,
    )

    total_loss = (
        alpha * hard_loss
        + (1 - alpha) * soft_loss
    )

    return hard_loss, soft_loss, total_loss


## 11. Calculate the Losses

This is one small batch demonstration, not a controlled experiment result.


In [11]:
temperature = 2.0
alpha = 0.7

hard_loss, soft_loss, total_loss = knowledge_distillation_loss(
    student_logits=student_logits,
    teacher_logits=teacher_logits,
    true_labels=labels,
    temperature=temperature,
    alpha=alpha,
)

print("temperature:", temperature)
print("alpha:", alpha)
print("hard-label loss:", round(hard_loss.item(), 4))
print("soft-target loss:", round(soft_loss.item(), 4))
print("combined KD loss:", round(total_loss.item(), 4))


temperature: 2.0
alpha: 0.7
hard-label loss: 2.3148
soft-target loss: 8.2443
combined KD loss: 4.0937


## 12. One Correct Student Update

Only the student optimizer receives parameters. We clone one student parameter and one teacher parameter before the update, then compare them afterward.


In [12]:
optimizer = Adam(student.parameters(), lr=0.001)

student_before = next(student.parameters()).detach().clone()
teacher_before = next(teacher.parameters()).detach().clone()

optimizer.zero_grad()
total_loss.backward()
optimizer.step()

student_after = next(student.parameters()).detach().clone()
teacher_after = next(teacher.parameters()).detach().clone()

student_updated = not torch.equal(student_before, student_after)
teacher_updated = not torch.equal(teacher_before, teacher_after)

print("student updated:", student_updated)
print("teacher updated:", teacher_updated)


student updated: True
teacher updated: False


## 13. Relation to the Training Script

This notebook shows the core idea on one batch. The reusable training implementation is in `scripts/train_mnist_distillation.py`.

That script repeats this process across many batches and epochs, saves outputs, and follows the project logging rules. This notebook is educational and is not a new experiment result.


## 14. Mini Exercise

After reading this notebook, try explaining these in your own words:

1. Why does the teacher use `eval()`?
2. Why do we freeze teacher parameters?
3. What does temperature change?
4. What does alpha control?
5. Why should only the student update?


## What I Learned

In this walkthrough, I learned that KD keeps the teacher fixed and trains only the student. The student learns from both true hard labels and teacher soft targets. Temperature controls how soft the teacher distribution becomes. The `T²` factor helps preserve the soft-loss gradient scale when temperature changes, and alpha balances hard-label learning with soft-target learning.
